# Part 1 Audit: adversarial + attr_reg recovery

This notebook audits the two fragile Part 1 models:

- `bert_adversarial`
- `bert_attr_reg`

It searches the workspace for artifacts, compares them with the original save format, inspects checkpoints when possible, and outputs a recovery verdict.


In [23]:
import json
from pathlib import Path

import pandas as pd
import torch

try:
    from safetensors.torch import load_file as load_safetensors_file
except Exception:
    load_safetensors_file = None

CURRENT = Path.cwd().resolve()
WORKSPACE = CURRENT if (CURRENT / 'city-swap-eval').exists() else CURRENT.parent
CITY_SWAP_REPO = WORKSPACE / 'city-swap-eval' if (WORKSPACE / 'city-swap-eval').exists() else CURRENT
PROJECT_REPO_CANDIDATES = [
    WORKSPACE / 'my-repository',
    CURRENT / 'my-repository',
    WORKSPACE / 'resume-screening',
    CURRENT / 'resume-screening',
    WORKSPACE,
]
RESUME_REPO = next((p for p in PROJECT_REPO_CANDIDATES if p.exists()), PROJECT_REPO_CANDIDATES[0])
NOTEBOOKS_CANDIDATES = [
    RESUME_REPO / 'resume-screening' / 'notebooks',
    RESUME_REPO / 'notebooks',
    WORKSPACE / 'resume-screening' / 'resume-screening' / 'notebooks',
    WORKSPACE / 'resume-screening' / 'notebooks',
    CURRENT / 'resume-screening' / 'resume-screening' / 'notebooks',
    CURRENT / 'resume-screening' / 'notebooks',
]
NOTEBOOKS_DIR = next((p for p in NOTEBOOKS_CANDIDATES if p.exists()), NOTEBOOKS_CANDIDATES[0])
OUTPUT_ROOT_CANDIDATES = [
    RESUME_REPO / 'notebooks' / 'results' / 'two-models-restore',
    RESUME_REPO / 'resume-screening' / 'notebooks' / 'results' / 'two-models-restore',
]
OUTPUT_ROOT = next((p for p in OUTPUT_ROOT_CANDIDATES if p.parent.exists()), OUTPUT_ROOT_CANDIDATES[0])
OUTPUT_DIR = OUTPUT_ROOT / 'model_recovery_audit'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Workspace     :', WORKSPACE)
print('City-swap repo:', CITY_SWAP_REPO)
print('Resume repo   :', RESUME_REPO)
print('Output dir    :', OUTPUT_DIR)


Workspace     : /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository
City-swap repo: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/city-swap-eval
Resume repo   : /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository
Output dir    : /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/two-models-restore/model_recovery_audit


In [24]:
MODEL_SPECS = {
    'adversarial': {
        'folder_name': 'bert_adversarial',
        'training_notebook': NOTEBOOKS_DIR / '61_adversarial_debiasing.ipynb',
        'expected_files': [
            'model.pt', 'classifier.pt', 'label_encoder.joblib', 'city_encoder.joblib',
            'training_config.json', 'tokenizer.json', 'tokenizer_config.json',
            'special_tokens_map.json', 'bert/config.json', 'bert/model.safetensors', 'bert/pytorch_model.bin',
        ],
        'best_checkpoint_names': ['bert_adversarial_best.pt'],
    },
    'attr_reg': {
        'folder_name': 'bert_attr_reg',
        'training_notebook': NOTEBOOKS_DIR / '63_attribution_regularization_v2.ipynb',
        'expected_files': [
            'model.pt', 'classifier.pt', 'label_encoder.joblib', 'training_config.json',
            'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json',
            'bert/config.json', 'bert/model.safetensors', 'bert/pytorch_model.bin',
        ],
        'best_checkpoint_names': ['bert_attr_reg_best.pt'],
    },
}

SEARCH_ROOTS = [
    CURRENT,
    CURRENT.parent,
    CURRENT.parent.parent,
    WORKSPACE,
    WORKSPACE.parent,
    RESUME_REPO,
    CITY_SWAP_REPO,
    WORKSPACE / '.tmp-repo-sync',
]
SEARCH_ROOTS = [p for p in SEARCH_ROOTS if p.exists()]
SEARCH_ROOTS


[PosixPath('/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks'),
 PosixPath('/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository'),
 PosixPath('/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS'),
 PosixPath('/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository'),
 PosixPath('/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS'),
 PosixPath('/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository'),
 PosixPath('/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/city-swap-eval')]

In [25]:
def safe_rel(path: Path) -> str:
    try:
        return str(path.relative_to(WORKSPACE))
    except Exception:
        return str(path)


def list_matches(name: str):
    matches = []
    for root in SEARCH_ROOTS:
        try:
            matches.extend(sorted(root.rglob(name)))
        except Exception:
            pass
    unique = []
    seen = set()
    for path in matches:
        key = str(path.resolve())
        if key not in seen:
            unique.append(path)
            seen.add(key)
    return unique


def find_weight_file(folder: Path):
    for name in ['model.safetensors', 'pytorch_model.bin', 'model.bin']:
        candidate = folder / name
        if candidate.exists():
            return candidate
    return None


def load_state_dict_any(path: Path):
    if path.suffix == '.safetensors':
        if load_safetensors_file is None:
            raise RuntimeError('safetensors is not available')
        return load_safetensors_file(str(path), device='cpu')
    obj = torch.load(path, map_location='cpu')
    if isinstance(obj, dict):
        for key in ['state_dict', 'model_state_dict', 'classifier_state_dict']:
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
        return obj
    if hasattr(obj, 'state_dict'):
        return obj.state_dict()
    raise TypeError(f'Unsupported checkpoint object type: {type(obj)}')


def inspect_state_dict(path: Path):
    result = {
        'exists': path.exists(),
        'path': safe_rel(path),
        'num_keys': None,
        'sample_keys': [],
        'classifier_keys': [],
        'error': '',
    }
    if not path.exists():
        return result
    try:
        state = load_state_dict_any(path)
        keys = sorted(state.keys())
        result['num_keys'] = len(keys)
        result['sample_keys'] = keys[:20]
        result['classifier_keys'] = [key for key in keys if key.startswith('classifier.') or key in {'weight', 'bias'}]
    except Exception as exc:
        result['error'] = f'{type(exc).__name__}: {exc}'
    return result


def resolve_existing_path(path: Path):
    candidates = [
        path,
        NOTEBOOKS_DIR / path.name,
        RESUME_REPO / 'resume-screening' / 'notebooks' / path.name,
        RESUME_REPO / 'notebooks' / path.name,
        WORKSPACE / 'resume-screening' / 'resume-screening' / 'notebooks' / path.name,
        WORKSPACE / 'resume-screening' / 'notebooks' / path.name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    matches = list_matches(path.name)
    for match in matches:
        if match.name == path.name and match.suffix == '.ipynb':
            return match
    checked = '\n'.join(str(c) for c in candidates)
    raise FileNotFoundError(f'Notebook not found: {path.name}. Checked:\n{checked}')


def read_notebook_source(path: Path) -> str:
    resolved = resolve_existing_path(path)
    nb = json.loads(resolved.read_text())
    parts = []
    for cell in nb.get('cells', []):
        if cell.get('cell_type') == 'code':
            parts.append(''.join(cell.get('source', [])))
    return '\n\n'.join(parts)


def extract_save_hints(source: str):
    return {
        'uses_save_pretrained_bert': 'save_pretrained(SAVE_DIR / "bert")' in source or "save_pretrained(SAVE_DIR / 'bert')" in source,
        'saves_classifier_pt': 'classifier.pt' in source,
        'saves_model_pt': 'model.pt' in source,
        'saves_best_pt': '_best.pt' in source,
        'saves_label_encoder': 'label_encoder.joblib' in source,
        'saves_city_encoder': 'city_encoder.joblib' in source,
    }


In [26]:
def audit_model(spec_name: str, spec: dict):
    folder_name = spec['folder_name']
    notebook_source = read_notebook_source(spec['training_notebook'])
    save_hints = extract_save_hints(notebook_source)

    folder_matches = list_matches(folder_name)
    best_matches = []
    for checkpoint_name in spec['best_checkpoint_names']:
        best_matches.extend(list_matches(checkpoint_name))

    expected_rows = []
    folder_match = folder_matches[0] if folder_matches else None
    for rel_name in spec['expected_files']:
        explicit_matches = list_matches(Path(rel_name).name)
        expected_path = (folder_match / rel_name) if folder_match is not None else (Path(folder_name) / rel_name)
        expected_rows.append({
            'model': spec_name,
            'expected_file': rel_name,
            'expected_under_found_folder': safe_rel(expected_path),
            'exists_under_found_folder': bool(folder_match is not None and expected_path.exists()),
            'matching_name_anywhere': '; '.join(safe_rel(match) for match in explicit_matches[:10]),
        })

    folder_details = []
    for folder in folder_matches:
        folder = Path(folder)
        bert_dir = folder / 'bert'
        config_path = bert_dir / 'config.json'
        bert_weights = find_weight_file(bert_dir)
        classifier_pt = folder / 'classifier.pt'
        model_pt = folder / 'model.pt'
        label_encoder = folder / 'label_encoder.joblib'
        city_encoder = folder / 'city_encoder.joblib'
        tokenizer_json = folder / 'tokenizer.json'
        reconstructable_hf = bert_dir.exists() and config_path.exists() and (bert_weights is not None) and classifier_pt.exists() and label_encoder.exists()
        folder_details.append({
            'model': spec_name,
            'folder': safe_rel(folder),
            'bert_dir_exists': bert_dir.exists(),
            'bert_config_exists': config_path.exists(),
            'bert_weight_file': safe_rel(bert_weights) if bert_weights else '',
            'classifier_pt_exists': classifier_pt.exists(),
            'model_pt_exists': model_pt.exists(),
            'label_encoder_exists': label_encoder.exists(),
            'city_encoder_exists': city_encoder.exists(),
            'tokenizer_json_exists': tokenizer_json.exists(),
            'reconstructable_hf': reconstructable_hf,
        })

    checkpoint_rows = [{'model': spec_name, **inspect_state_dict(Path(match))} for match in best_matches]
    classifier_rows = []
    for folder in folder_matches:
        classifier_pt = Path(folder) / 'classifier.pt'
        if classifier_pt.exists():
            classifier_rows.append({'model': spec_name, **inspect_state_dict(classifier_pt)})

    verdict = 'missing_everything'
    if folder_details and any(row['reconstructable_hf'] for row in folder_details):
        verdict = 'can_reconstruct_hf'
    elif folder_details:
        verdict = 'partial_folder_found'
    elif checkpoint_rows:
        verdict = 'only_raw_checkpoint_found'

    summary = {
        'model': spec_name,
        'folder_name': folder_name,
        'training_notebook': safe_rel(spec['training_notebook']),
        'verdict': verdict,
        'folder_matches': [safe_rel(Path(p)) for p in folder_matches],
        'best_checkpoint_matches': [safe_rel(Path(p)) for p in best_matches],
        **save_hints,
    }
    return summary, pd.DataFrame(expected_rows), pd.DataFrame(folder_details), pd.DataFrame(checkpoint_rows), pd.DataFrame(classifier_rows)


In [27]:
all_summaries = []
all_expected = []
all_folders = []
all_checkpoints = []
all_classifiers = []

for model_name, spec in MODEL_SPECS.items():
    summary, expected_df, folder_df, checkpoint_df, classifier_df = audit_model(model_name, spec)
    all_summaries.append(summary)
    all_expected.append(expected_df)
    all_folders.append(folder_df)
    all_checkpoints.append(checkpoint_df)
    all_classifiers.append(classifier_df)

summary_df = pd.DataFrame(all_summaries)
expected_df = pd.concat([df for df in all_expected if not df.empty], ignore_index=True)
folder_df = pd.concat([df for df in all_folders if not df.empty], ignore_index=True) if any(not df.empty for df in all_folders) else pd.DataFrame()
checkpoint_df = pd.concat([df for df in all_checkpoints if not df.empty], ignore_index=True) if any(not df.empty for df in all_checkpoints) else pd.DataFrame()
classifier_df = pd.concat([df for df in all_classifiers if not df.empty], ignore_index=True) if any(not df.empty for df in all_classifiers) else pd.DataFrame()

summary_df.to_csv(OUTPUT_DIR / 'recovery_summary.csv', index=False)
expected_df.to_csv(OUTPUT_DIR / 'expected_files_audit.csv', index=False)
if not folder_df.empty:
    folder_df.to_csv(OUTPUT_DIR / 'folder_details.csv', index=False)
if not checkpoint_df.empty:
    checkpoint_df.to_csv(OUTPUT_DIR / 'checkpoint_details.csv', index=False)
if not classifier_df.empty:
    classifier_df.to_csv(OUTPUT_DIR / 'classifier_details.csv', index=False)

summary_df


,model,folder_name,training_notebook,verdict,folder_matches,best_checkpoint_matches,uses_save_pretrained_bert,saves_classifier_pt,saves_model_pt,saves_best_pt,saves_label_encoder,saves_city_encoder
0,adversarial,bert_adversarial,notebooks/61_adversarial_debiasing.ipynb,can_reconstruct_hf,[notebooks/models/bert_adversarial],[notebooks/models/bert_adversarial_best.pt],True,True,True,True,True,True
1,attr_reg,bert_attr_reg,notebooks/63_attribution_regularization_v2.ipynb,can_reconstruct_hf,[notebooks/models/bert_attr_reg],[notebooks/models/bert_attr_reg_best.pt],True,True,True,True,True,False


In [28]:
print('=== Summary ===')
display(summary_df)
print('\n=== Expected files audit ===')
display(expected_df)
if not folder_df.empty:
    print('\n=== Folder details ===')
    display(folder_df)
if not checkpoint_df.empty:
    print('\n=== Checkpoint details ===')
    display(checkpoint_df[['model', 'path', 'num_keys', 'classifier_keys', 'error']])
if not classifier_df.empty:
    print('\n=== Classifier sidecars ===')
    display(classifier_df[['model', 'path', 'num_keys', 'classifier_keys', 'error']])


=== Summary ===


,model,folder_name,training_notebook,verdict,folder_matches,best_checkpoint_matches,uses_save_pretrained_bert,saves_classifier_pt,saves_model_pt,saves_best_pt,saves_label_encoder,saves_city_encoder
0,adversarial,bert_adversarial,notebooks/61_adversarial_debiasing.ipynb,can_reconstruct_hf,[notebooks/models/bert_adversarial],[notebooks/models/bert_adversarial_best.pt],True,True,True,True,True,True
1,attr_reg,bert_attr_reg,notebooks/63_attribution_regularization_v2.ipynb,can_reconstruct_hf,[notebooks/models/bert_attr_reg],[notebooks/models/bert_attr_reg_best.pt],True,True,True,True,True,False



=== Expected files audit ===


,model,expected_file,expected_under_found_folder,exists_under_found_folder,matching_name_anywhere
0,adversarial,model.pt,notebooks/models/bert_adversarial/model.pt,True,notebooks/models/bert_adversarial/model.pt; no...
1,adversarial,classifier.pt,notebooks/models/bert_adversarial/classifier.pt,True,notebooks/models/bert_adversarial/classifier.p...
2,adversarial,label_encoder.joblib,notebooks/models/bert_adversarial/label_encode...,True,notebooks/models/bert_9classes_final/label_enc...
3,adversarial,city_encoder.joblib,notebooks/models/bert_adversarial/city_encoder...,True,notebooks/models/bert_adversarial/city_encoder...
4,adversarial,training_config.json,notebooks/models/bert_adversarial/training_con...,True,notebooks/models/bert_9classes_final/training_...
5,adversarial,tokenizer.json,notebooks/models/bert_adversarial/tokenizer.json,True,notebooks/models/bert_9classes_final/tokenizer...
6,adversarial,tokenizer_config.json,notebooks/models/bert_adversarial/tokenizer_co...,True,notebooks/models/bert_9classes_final/tokenizer...
7,adversarial,special_tokens_map.json,notebooks/models/bert_adversarial/special_toke...,False,archive/models/bert_9classes_gdro_eta0.1_1ep/s...
8,adversarial,bert/config.json,notebooks/models/bert_adversarial/bert/config....,True,notebooks/models/bert_9classes_final/config.js...
9,adversarial,bert/model.safetensors,notebooks/models/bert_adversarial/bert/model.s...,True,notebooks/models/bert_9classes_final/model.saf...



=== Folder details ===


,model,folder,bert_dir_exists,bert_config_exists,bert_weight_file,classifier_pt_exists,model_pt_exists,label_encoder_exists,city_encoder_exists,tokenizer_json_exists,reconstructable_hf
0,adversarial,notebooks/models/bert_adversarial,True,True,notebooks/models/bert_adversarial/bert/model.s...,True,True,True,True,True,True
1,attr_reg,notebooks/models/bert_attr_reg,True,True,notebooks/models/bert_attr_reg/bert/model.safe...,True,True,True,False,True,True



=== Checkpoint details ===


,model,path,num_keys,classifier_keys,error
0,adversarial,notebooks/models/bert_adversarial_best.pt,205,"[classifier.1.bias, classifier.1.weight]",
1,attr_reg,notebooks/models/bert_attr_reg_best.pt,201,"[classifier.1.bias, classifier.1.weight]",



=== Classifier sidecars ===


,model,path,num_keys,classifier_keys,error
0,adversarial,notebooks/models/bert_adversarial/classifier.pt,2,[],
1,attr_reg,notebooks/models/bert_attr_reg/classifier.pt,2,[],


In [29]:
def _stringify(value):
    if isinstance(value, list):
        return ', '.join(str(x) for x in value) if value else '-'
    try:
        if pd.isna(value):
            return '-'
    except Exception:
        pass
    text = str(value)
    return text if text else '-'


def format_summary_text(summary_df):
    lines = ['=== Recovery Summary ===']
    for _, row in summary_df.iterrows():
        lines.extend([
            f"model: {row['model']}",
            f"folder_name: {row['folder_name']}",
            f"verdict: {row['verdict']}",
            f"training_notebook: {row['training_notebook']}",
            f"folder_matches: {_stringify(row['folder_matches'])}",
            f"best_checkpoint_matches: {_stringify(row['best_checkpoint_matches'])}",
            f"save_hints: bert={row['uses_save_pretrained_bert']}, classifier_pt={row['saves_classifier_pt']}, model_pt={row['saves_model_pt']}, best_pt={row['saves_best_pt']}, label_encoder={row['saves_label_encoder']}, city_encoder={row['saves_city_encoder']}",
            '-' * 60,
        ])
    return '\n'.join(lines)


def format_checkpoint_text(checkpoint_df, title='=== Checkpoint Details ==='):
    lines = [title]
    if checkpoint_df.empty:
        lines.append('no rows')
        return '\n'.join(lines)
    for _, row in checkpoint_df.iterrows():
        lines.extend([
            f"model: {row['model']}",
            f"path: {row['path']}",
            f"num_keys: {_stringify(row['num_keys'])}",
            f"classifier_keys: {_stringify(row['classifier_keys'])}",
            f"error: {_stringify(row['error'])}",
            '-' * 60,
        ])
    return '\n'.join(lines)


summary_text = format_summary_text(summary_df)
checkpoint_text = format_checkpoint_text(checkpoint_df)
classifier_text = format_checkpoint_text(classifier_df, title='=== Classifier Sidecars ===')

(OUTPUT_DIR / 'recovery_summary.txt').write_text(summary_text)
(OUTPUT_DIR / 'checkpoint_details.txt').write_text(checkpoint_text)
(OUTPUT_DIR / 'classifier_details.txt').write_text(classifier_text)

print(summary_text)
print()
print(checkpoint_text)
if not classifier_df.empty:
    print()
    print(classifier_text)


=== Recovery Summary ===
model: adversarial
folder_name: bert_adversarial
verdict: can_reconstruct_hf
training_notebook: notebooks/61_adversarial_debiasing.ipynb
folder_matches: notebooks/models/bert_adversarial
best_checkpoint_matches: notebooks/models/bert_adversarial_best.pt
save_hints: bert=True, classifier_pt=True, model_pt=True, best_pt=True, label_encoder=True, city_encoder=True
------------------------------------------------------------
model: attr_reg
folder_name: bert_attr_reg
verdict: can_reconstruct_hf
training_notebook: notebooks/63_attribution_regularization_v2.ipynb
folder_matches: notebooks/models/bert_attr_reg
best_checkpoint_matches: notebooks/models/bert_attr_reg_best.pt
save_hints: bert=True, classifier_pt=True, model_pt=True, best_pt=True, label_encoder=True, city_encoder=False
------------------------------------------------------------

=== Checkpoint Details ===
model: adversarial
path: notebooks/models/bert_adversarial_best.pt
num_keys: 205
classifier_keys: cl

In [30]:
def make_recovery_todo(summary_df, folder_df):
    rows = []
    folder_df = folder_df if not folder_df.empty else pd.DataFrame()
    for _, row in summary_df.iterrows():
        model = row['model']
        verdict = row['verdict']
        missing_bits = []
        if not folder_df.empty:
            subset = folder_df[folder_df['model'] == model]
            if not subset.empty:
                best = subset.iloc[0]
                if not bool(best.get('bert_dir_exists', False)):
                    missing_bits.append('bert backbone folder')
                if not bool(best.get('classifier_pt_exists', False)):
                    missing_bits.append('classifier.pt')
                if not bool(best.get('label_encoder_exists', False)):
                    missing_bits.append('label_encoder.joblib')
                if not bool(best.get('tokenizer_json_exists', False)):
                    missing_bits.append('tokenizer files')
        if not missing_bits and verdict in {'missing_everything', 'only_raw_checkpoint_found'}:
            missing_bits = ['model folder', 'tokenizer files', 'label encoder']

        if verdict == 'can_reconstruct_hf':
            action = 'Build a temporary HF folder from bert/ + classifier.pt + tokenizer + label encoder.'
        elif verdict == 'partial_folder_found':
            action = 'Folder is incomplete. Restore missing pieces from backup or source machine.'
        elif verdict == 'only_raw_checkpoint_found':
            action = 'Need architecture-aware restoration plus tokenizer and label encoder.'
        else:
            action = 'No usable artifacts found in this workspace. Pull/copy the saved model folder or raw checkpoints from the original machine.'

        rows.append({
            'model': model,
            'verdict': verdict,
            'missing_or_needed': '; '.join(missing_bits),
            'next_action': action,
        })
    return pd.DataFrame(rows)


todo_df = make_recovery_todo(summary_df, folder_df)
todo_df.to_csv(OUTPUT_DIR / 'recovery_todo.csv', index=False)
todo_df


,model,verdict,missing_or_needed,next_action
0,adversarial,can_reconstruct_hf,,Build a temporary HF folder from bert/ + class...
1,attr_reg,can_reconstruct_hf,,Build a temporary HF folder from bert/ + class...
